In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    brier_score_loss
)
from lightgbm import LGBMClassifier


# ============================================================
# Config
# ============================================================
INPUT_PATH = Path("./training_data_standardization_with_pairwise_products.csv")
LABEL_COL = "label"

SEEDS = [35, 42, 55, 64, 100]

TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2

ECE_BINS = 15
N_ADDITIONAL_SELECT = 34

OUT_METRICS = Path("./lgbm_selected47_metrics.csv")
OUT_FEATURES_WIDE = Path("./lgbm_selected47_features_wide.csv")
OUT_FEATURES_GROUPED = Path("./lgbm_selected47_features_grouped.csv")
OUT_SELECTED_IMPORTANCE = Path("./lgbm_selected47_selected_pairwise_importance.csv")


# ============================================================
# Utils
# ============================================================
def remove_index_like_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    index처럼 의미 없는 컬럼 제거
    예:
    - Unnamed: 0
    - index
    - 값이 0..n-1 또는 1..n 인 컬럼
    """
    df = df.copy()
    drop_cols = []

    for col in df.columns:
        col_lower = str(col).strip().lower()

        if col_lower.startswith("unnamed") or col_lower in ["index", "level_0"]:
            drop_cols.append(col)
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            values = df[col].values
            seq0 = np.arange(len(df))
            seq1 = np.arange(1, len(df) + 1)

            if np.array_equal(values, seq0) or np.array_equal(values, seq1):
                drop_cols.append(col)

    if drop_cols:
        print("제거된 index-like 컬럼:", drop_cols)
        df = df.drop(columns=drop_cols)

    return df


def expected_calibration_error(y_true, y_prob, n_bins=15):
    """
    ECE 계산
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    n = len(y_true)

    for b in range(n_bins):
        mask = (bin_ids == b)
        if np.sum(mask) == 0:
            continue

        bin_acc = np.mean(y_true[mask])
        bin_conf = np.mean(y_prob[mask])
        bin_weight = np.sum(mask) / n

        ece += np.abs(bin_acc - bin_conf) * bin_weight

    return float(ece)


def find_best_threshold_by_f1(y_true, y_prob):
    """
    validation set에서 F1이 최대가 되는 threshold 선택
    """
    thresholds = np.linspace(0.01, 0.99, 99)

    best_threshold = 0.5
    best_f1 = -1.0

    for th in thresholds:
        y_pred = (y_prob >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_threshold = th

    return float(best_threshold), float(best_f1)


def evaluate_binary(y_true, y_prob, threshold, ece_bins=15):
    """
    AUROC, AUPRC, F1, Brier, ECE 계산
    """
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier": brier_score_loss(y_true, y_prob),
        "ECE": expected_calibration_error(y_true, y_prob, n_bins=ece_bins)
    }


def build_lgbm_model(seed, scale_pos_weight=1.0):
    """
    randomness가 실제 반영되도록 subsample / colsample_bytree 포함
    """
    model = LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=0.0,
        random_state=seed,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        importance_type="gain",
        verbosity=-1
    )
    return model


# ============================================================
# Load
# ============================================================
df = pd.read_csv(INPUT_PATH)
df = remove_index_like_columns(df)

if LABEL_COL not in df.columns:
    raise ValueError(f"'{LABEL_COL}' 컬럼이 없습니다.")

df = df.dropna(subset=[LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# feature / target
feature_cols = [c for c in df.columns if c != LABEL_COL]

# 원본 / 조합 feature 분리
original_features = [c for c in feature_cols if "__mul__" not in c]
pairwise_features = [c for c in feature_cols if "__mul__" in c]

print(f"원본 feature 수: {len(original_features)}")
print(f"pairwise feature 수: {len(pairwise_features)}")
print(f"전체 feature 수: {len(feature_cols)}")

if len(original_features) != 13:
    print(f"[주의] 원본 feature 수가 13이 아닙니다: {len(original_features)}")

if len(pairwise_features) < N_ADDITIONAL_SELECT:
    raise ValueError(
        f"선택할 pairwise feature 수({N_ADDITIONAL_SELECT})보다 "
        f"실제 pairwise feature 수({len(pairwise_features)})가 적습니다."
    )

X = df[feature_cols].copy()
y = df[LABEL_COL].copy()

# 결측 처리
if X.isnull().sum().sum() > 0:
    print("feature 내 결측치 발견 -> median 대체")
    X = X.fillna(X.median(numeric_only=True))

# 비수치형 처리
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    print("비수치형 컬럼 factorize:", non_numeric_cols)
    for col in non_numeric_cols:
        X[col] = pd.factorize(X[col])[0]

X = X.astype(float)

print(f"전체 데이터 shape: {df.shape}")
print(f"label 분포:\n{y.value_counts(dropna=False).sort_index()}")


# ============================================================
# Split (고정)
# ============================================================
n = len(df)

train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val = X.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()

X_test = X.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

print("\n[Split]")
print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Val  : {X_val.shape}, {y_val.shape}")
print(f"Test : {X_test.shape}, {y_test.shape}")

pos_count = int((y_train == 1).sum())
neg_count = int((y_train == 0).sum())

if pos_count == 0:
    raise ValueError("train set에 positive class가 없습니다.")

scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0


# ============================================================
# Seed별 feature selection + final training
# ============================================================
results = []
selected_feature_dict = {}
grouped_feature_rows = []
selected_importance_rows = []

for seed in SEEDS:
    print(f"\n==============================")
    print(f"[SEED {seed}] start")
    print(f"==============================")

    # --------------------------------------------------------
    # 1) 전체 feature로 1차 LightGBM 학습 -> pairwise importance 추출
    # --------------------------------------------------------
    fs_model = build_lgbm_model(seed=seed, scale_pos_weight=scale_pos_weight)
    fs_model.fit(X_train, y_train)

    importances = fs_model.booster_.feature_importance(importance_type="gain")
    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance_gain": importances
    })

    pairwise_importance_df = importance_df[
        importance_df["feature"].isin(pairwise_features)
    ].copy()

    pairwise_importance_df = pairwise_importance_df.sort_values(
        "importance_gain", ascending=False
    ).reset_index(drop=True)

    selected_pairwise = pairwise_importance_df.head(N_ADDITIONAL_SELECT)["feature"].tolist()

    if len(selected_pairwise) < N_ADDITIONAL_SELECT:
        raise ValueError(
            f"[SEED {seed}] 선택된 pairwise feature 수가 부족합니다: "
            f"{len(selected_pairwise)}"
        )

    # 선택된 pairwise importance 저장
    selected_pairwise_df = pairwise_importance_df.head(N_ADDITIONAL_SELECT).copy()
    selected_pairwise_df["seed"] = seed
    selected_pairwise_df["rank_within_seed"] = np.arange(1, len(selected_pairwise_df) + 1)
    selected_importance_rows.append(
        selected_pairwise_df[["seed", "rank_within_seed", "feature", "importance_gain"]]
    )

    # --------------------------------------------------------
    # 2) 최종 47개 feature 구성
    # --------------------------------------------------------
    final_features = original_features + selected_pairwise
    selected_feature_dict[seed] = final_features

    print(f"[SEED {seed}] selected pairwise feature 수: {len(selected_pairwise)}")
    print(f"[SEED {seed}] final feature 수: {len(final_features)}")

    # grouped feature 저장용
    grouped_feature_rows.append({
        "seed": seed,
        "type": "original",
        "feature_count": len(original_features),
        "features": ", ".join(original_features)
    })
    grouped_feature_rows.append({
        "seed": seed,
        "type": "pairwise",
        "feature_count": len(selected_pairwise),
        "features": ", ".join(selected_pairwise)
    })

    # --------------------------------------------------------
    # 3) 최종 47개 feature로 다시 학습
    # --------------------------------------------------------
    final_model = build_lgbm_model(seed=seed, scale_pos_weight=scale_pos_weight)
    final_model.fit(X_train[final_features], y_train)

    # threshold 선택: validation F1 최대
    val_prob = final_model.predict_proba(X_val[final_features])[:, 1]
    best_threshold, best_val_f1 = find_best_threshold_by_f1(y_val.values, val_prob)

    # test 평가
    test_prob = final_model.predict_proba(X_test[final_features])[:, 1]
    test_metrics = evaluate_binary(
        y_true=y_test.values,
        y_prob=test_prob,
        threshold=best_threshold,
        ece_bins=ECE_BINS
    )

    result_row = {
        "seed": seed,
        "n_original_features": len(original_features),
        "n_selected_pairwise": len(selected_pairwise),
        "n_final_features": len(final_features),
        "threshold": best_threshold,
        "val_best_f1": best_val_f1,
        "AUROC": test_metrics["AUROC"],
        "AUPRC": test_metrics["AUPRC"],
        "F1": test_metrics["F1"],
        "Brier": test_metrics["Brier"],
        "ECE": test_metrics["ECE"]
    }
    results.append(result_row)

    print(f"[SEED {seed}] threshold: {best_threshold:.4f}")
    print(f"[SEED {seed}] AUROC: {test_metrics['AUROC']:.6f}")
    print(f"[SEED {seed}] AUPRC: {test_metrics['AUPRC']:.6f}")
    print(f"[SEED {seed}] F1   : {test_metrics['F1']:.6f}")
    print(f"[SEED {seed}] Brier: {test_metrics['Brier']:.6f}")
    print(f"[SEED {seed}] ECE  : {test_metrics['ECE']:.6f}")


# ============================================================
# Metrics 저장용 DataFrame + mean row
# ============================================================
df_results = pd.DataFrame(results)

mean_row = {}
for col in df_results.columns:
    if col == "seed":
        mean_row[col] = "mean"
    elif pd.api.types.is_numeric_dtype(df_results[col]):
        mean_row[col] = df_results[col].mean()
    else:
        mean_row[col] = ""

df_results = pd.concat(
    [df_results, pd.DataFrame([mean_row])],
    ignore_index=True
)


# ============================================================
# Feature wide format 저장
# - 한 seed당 한 row
# - feature_1 ~ feature_47
# ============================================================
feature_wide_rows = []

for seed in SEEDS:
    feat_list = selected_feature_dict[seed]
    row = {"seed": seed}

    for i, feat in enumerate(feat_list, start=1):
        row[f"feature_{i}"] = feat

    feature_wide_rows.append(row)

df_features_wide = pd.DataFrame(feature_wide_rows)


# ============================================================
# Grouped feature 저장
# - 한 seed당 original / pairwise 분리
# ============================================================
df_features_grouped = pd.DataFrame(grouped_feature_rows)


# ============================================================
# Selected pairwise importance 저장
# ============================================================
df_selected_importance = pd.concat(selected_importance_rows, axis=0, ignore_index=True)


# ============================================================
# Save
# ============================================================
df_results.to_csv(OUT_METRICS, index=False, encoding="utf-8-sig")
df_features_wide.to_csv(OUT_FEATURES_WIDE, index=False, encoding="utf-8-sig")
df_features_grouped.to_csv(OUT_FEATURES_GROUPED, index=False, encoding="utf-8-sig")
df_selected_importance.to_csv(OUT_SELECTED_IMPORTANCE, index=False, encoding="utf-8-sig")


# ============================================================
# Print summary
# ============================================================
print("\n====================================")
print("저장 완료")
print("====================================")
print(f"1) Metrics CSV                : {OUT_METRICS.resolve()}")
print(f"2) Features Wide CSV          : {OUT_FEATURES_WIDE.resolve()}")
print(f"3) Features Grouped CSV       : {OUT_FEATURES_GROUPED.resolve()}")
print(f"4) Selected Importance CSV    : {OUT_SELECTED_IMPORTANCE.resolve()}")

print("\n=== Metrics Preview ===")
print(df_results)

print("\n=== Features Wide Preview ===")
print(df_features_wide.head())

print("\n=== Features Grouped Preview ===")
print(df_features_grouped.head())

print("\n=== Selected Pairwise Importance Preview ===")
print(df_selected_importance.head(10))

원본 feature 수: 13
pairwise feature 수: 78
전체 feature 수: 91
전체 데이터 shape: (17881, 92)
label 분포:
label
0    17485
1      396
Name: count, dtype: int64

[Split]
Train: (12516, 91), (12516,)
Val  : (1788, 91), (1788,)
Test : (3577, 91), (3577,)

[SEED 35] start
[SEED 35] selected pairwise feature 수: 34
[SEED 35] final feature 수: 47
[SEED 35] threshold: 0.7500
[SEED 35] AUROC: 0.827074
[SEED 35] AUPRC: 0.348149
[SEED 35] F1   : 0.366972
[SEED 35] Brier: 0.037630
[SEED 35] ECE  : 0.035915

[SEED 42] start
[SEED 42] selected pairwise feature 수: 34
[SEED 42] final feature 수: 47
[SEED 42] threshold: 0.8900
[SEED 42] AUROC: 0.835605
[SEED 42] AUPRC: 0.355255
[SEED 42] F1   : 0.381443
[SEED 42] Brier: 0.036898
[SEED 42] ECE  : 0.034988

[SEED 55] start
[SEED 55] selected pairwise feature 수: 34
[SEED 55] final feature 수: 47
[SEED 55] threshold: 0.9000
[SEED 55] AUROC: 0.835458
[SEED 55] AUPRC: 0.339638
[SEED 55] F1   : 0.345946
[SEED 55] Brier: 0.037359
[SEED 55] ECE  : 0.034299

[SEED 64] start
[SE